# Bell states and correlations
**Quantum Computing · Week 4 (Entanglement) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w4d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

The four Bell states, their correlations in three bases, and a test that no pair of ordinary coins can pass.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.primitives import StatevectorSampler

def bell(name):
    """H + CNOT makes |Phi+>; an X makes the Psi states, a Z puts in the minus sign."""
    qc = QuantumCircuit(2)
    if name in ("Psi+", "Psi-"):
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    if name in ("Phi-", "Psi-"):
        qc.z(0)
    return qc

for name in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    print(f"{name}: amplitudes (00, 01, 10, 11) = {np.round(Statevector(bell(name)).data.real, 3) + 0}")

## Correlations in three bases
`expectation_value` of $ZZ$, $XX$ and $YY$: +1 means the two qubits always agree in that basis, -1 always disagree.

In [ ]:
print("        ZZ     XX     YY")
for name in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    sv = Statevector(bell(name))
    vals = [sv.expectation_value(SparsePauliOp(p)).real for p in ["ZZ", "XX", "YY"]]
    print(f"{name}:  " + "  ".join(f"{v:+.2f}" for v in vals))

Each Bell state has its own pattern of signs: perfect agreement or disagreement in **every** basis.
Yet each qubit on its own is a perfect coin: its reduced state is the center of the Bloch ball.

In [ ]:
rho_q0 = partial_trace(Statevector(bell("Phi+")), [1])     # forget qubit 1, keep qubit 0
print("qubit 0 alone:\n", np.round(rho_q0.data.real, 3))
print("its Bloch vector:", [round(float(np.real(rho_q0.expectation_value(SparsePauliOp(p)))), 3) for p in "XYZ"])

## The CHSH test
Measure qubit 0 along angle $a$ or $a'$ and qubit 1 along $b$ or $b'$ (angles in the x-z plane), and combine
the four correlations: $S = E(a,b) + E(a,b') + E(a',b) - E(a',b')$. Any classical explanation, with the
outcomes decided in advance, gives $|S| \le 2$. Quantum mechanics allows up to $2\sqrt2 \approx 2.83$.

In [ ]:
sampler = StatevectorSampler(seed=4)

def E(theta_a, theta_b, shots=20000):
    qc = bell("Phi+")
    qc.ry(-theta_a, 0)          # turn so that "measure along theta" becomes "measure Z"
    qc.ry(-theta_b, 1)
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    agree = counts.get("00", 0) + counts.get("11", 0)
    return (2 * agree - shots) / shots

def E_exact(theta_a, theta_b):
    qc = bell("Phi+")
    qc.ry(-theta_a, 0)
    qc.ry(-theta_b, 1)
    p = Statevector(qc).probabilities()            # order 00, 01, 10, 11
    return p[0] - p[1] - p[2] + p[3]

a, a2, b, b2 = 0, np.pi / 2, np.pi / 4, -np.pi / 4
S_shots = E(a, b) + E(a, b2) + E(a2, b) - E(a2, b2)
S_exact = E_exact(a, b) + E_exact(a, b2) + E_exact(a2, b) - E_exact(a2, b2)
print(f"S from 4 x 20000 shots = {S_shots:.3f}  (shot noise is about ±0.01)")
print(f"S exact                = {S_exact:.3f}  = 2·sqrt(2)")
print("Any classical explanation gives |S| <= 2: the Bell state breaks that limit.")

## Your turn: name that Bell state
`mystery()` makes one of the four Bell states. Measure $\langle ZZ\rangle$ and $\langle XX\rangle$ with shots
(change the basis on **both** qubits for XX) and use the sign table above to name it.

In [ ]:
_secret = ["Phi+", "Phi-", "Psi+", "Psi-"][np.random.default_rng(8).integers(4)]

def mystery():
    return bell(_secret)

def correlation(basis, shots=4000):
    qc = mystery()
    # TODO: for basis "X", add an H on both qubits before measuring
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    agree = counts.get("00", 0) + counts.get("11", 0)
    return (2 * agree - shots) / shots

print("ZZ:", correlation("Z"), "  XX:", correlation("X"))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def correlation(basis, shots=4000):
    qc = mystery()
    if basis == "X":
        qc.h([0, 1])
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    agree = counts.get("00", 0) + counts.get("11", 0)
    return (2 * agree - shots) / shots

zz, xx = correlation("Z"), correlation("X")
table = {(1, 1): "Phi+", (1, -1): "Phi-", (-1, 1): "Psi+", (-1, -1): "Psi-"}
guess = table[(int(np.sign(zz)), int(np.sign(xx)))]
print(f"ZZ = {zz:+.2f}, XX = {xx:+.2f}  ->  it's {guess}   (the secret was {_secret})")